In [ ]:
import cv2
import numpy as np
from pathlib import Path
import matplotlib.pyplot as plt
import segmentation_models_pytorch as smp
import torch
from torchvision import transforms
from ImgTiler import SplitImage
from tqdm import tqdm

from utils_for_inference.image_proccesing import get_binary_mask, split_image_into_patch, reconstruct_from_patches_correct, resize_to_power_of_two_simple
from utils_for_inference.Model import ModelInferense
from utils_for_inference.utils import get_image_in_folder

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else 'cpu')
# device = 'cpu'
print(f"Device use: {device}")

In [ ]:
image_list = get_image_in_folder("./datasets/UAV/presentation")
model = ModelInferense("./model_weights/weights_2026-02-11T16:48:31.218593/last_Multiclass_Segmentation_Buildings_UAV_2026-02-14 16:25:42.610927_epoch:182.pth", device)

In [ ]:
for image_path in tqdm(image_list, total=len(image_list), desc="Image list extract", position=0, leave=True):
    print(image_path)
    image = cv2.imread(image_path)
    # image = cv2.resize(image, (8192, 8192), interpolation=cv2.INTER_LINEAR)
    image, _, _, _ = resize_to_power_of_two_simple(image_path)
    image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)

    filtered_x = split_image_into_patch(image, 2048)
    masks_filtered_x = []
    masked_region_filtered_x = []
    segmented_filtered_x = []
    for patch in tqdm(filtered_x, total=len(filtered_x), desc="Patch get predict", position=0, leave=True):
        predict = model.predict(patch)
        predict.to("cpu")
        torch.cuda.empty_cache()
        binary_mask, masked_region, segmented = get_binary_mask(patch, predict)
        masks_filtered_x.append(binary_mask)
        masked_region_filtered_x.append(masked_region)
        segmented_filtered_x.append(segmented)

    masks_reconstruct = reconstruct_from_patches_correct(masks_filtered_x, image.shape)
    masked_region_reconstruct = reconstruct_from_patches_correct(masked_region_filtered_x, image.shape)
    cv2.imwrite(f'./datasets/save results/mask_{image_path.name}', masked_region_reconstruct)
    segmented_reconstruct = reconstruct_from_patches_correct(segmented_filtered_x, image.shape)

In [ ]:
import shutil
import zipfile
import tarfile
def pack_folder_to_archive(folder_path, output_archive=None, format='zip'):

    folder_path = Path(folder_path)

    if output_archive is None:
        output_archive = folder_path.parent / f"{folder_path.name}.{format.split('tar')[0] + 'zip' if 'tar' in format else format}"
    
    archive_path = shutil.make_archive(
        base_name=str(output_archive).replace(f'.{format}', ''),
        format=format,
        root_dir=folder_path.parent,
        base_dir=folder_path.name
    )
    
    print(f"Папка '{folder_path}' успешно упакована в '{archive_path}'")
    return archive_path

archive_path = pack_folder_to_archive('./datasets/prepared_dataset', './multiclass_image.zip')